# 🩺 Huấn luyện Mamba-UNet (MambaVision Backbone) trên Google Colab

Quy trình chuẩn hóa toàn diện:
1. **Kiểm tra phần cứng GPU** (`!nvidia-smi`)
2. **Cài đặt môi trường PyTorch 2.6.0 (cu126) & Triton 3.3.1** (tương thích chính xác với C++ CUDA ABI=True)
3. **Mount Google Drive & Cài đặt 2 Wheels CUDA (`causal_conv1d` + `mamba_ssm`)** từ Drive
4. **Khởi động lại Runtime** để nạp kernel C++/CUDA vào hệ thống
5. **Kiểm tra xác minh CUDA Selective Scan** (fp32 + fp16, forward + backward)
6. **Đồng bộ mã nguồn từ GitHub**
7. **Giải nén dữ liệu từ Google Drive vào SSD cục bộ** (có tùy chọn số lượng ca bệnh nhân `NUM_PATIENTS`)
8. **Huấn luyện mô hình với đầy đủ tham số tùy chỉnh** (Backbone, Pretrained, Activation, Loss, LR, Workers, Resume...)
9. **Đánh giá 3D cấp độ bệnh nhân** trên tập Validation
10. **Lưu kết quả & Checkpoints vào Google Drive theo từng thư mục phiên chạy kèm bản báo cáo Markdown/JSON** (tự động append nếu Resume)

In [ ]:
# 1. Kiểm tra phần cứng và GPU khả dụng
!nvidia-smi

In [ ]:
# 2. Thiết lập môi trường PyTorch 2.6.0 (cu126) + Triton 3.3.1
# (Chạy ở tiến trình riêng, không import torch trực tiếp trong kernel này)
import subprocess, sys

def probe(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

TORCH_PROBE = "import torch;print(torch.__version__, int(torch._C._GLIBCXX_USE_CXX11_ABI))"
cur = probe(TORCH_PROBE)
print("Torch hiện tại:", cur)

if cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"):
    print("⏭️ Đã có torch 2.6.0+cu126 (ABI=True), bỏ qua bước cài torch.")
else:
    print("🧹 Dọn dẹp và cài đặt PyTorch 2.6.0+cu126...")
    !rm -rf /usr/local/lib/python3.13/dist-packages/~* 2>/dev/null || true
    !pip uninstall torch torchvision torchaudio causal-conv1d mamba-ssm -y -q 2>/dev/null || true
    !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126 -q
    cur = probe(TORCH_PROBE)
    print("Torch sau khi cài:", cur)

assert cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"), \
    f"❌ Torch chưa đúng (cần 2.6.0+cu126 và ABI=1, đang là: {cur}). Hãy thử chạy lại cell này."

print("\n⚙️ Cài đặt Triton 3.3.1 và các thư viện cần thiết (SimpleITK, etc.)...")
!python -m pip install triton==3.3.1 SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib -q
trt = probe("import triton;print(triton.__version__, hasattr(triton,'set_allocator'))")
print("Triton:", trt)
assert trt and trt.endswith("True"), f"❌ Triton chưa có set_allocator (đang là: {trt})"

print("\n✅ CELL 2 XONG. Chạy tiếp cell 3 để mount Drive và cài 2 wheels.")

In [ ]:
# 3. Mount Google Drive & Cài đặt 2 Wheel CUDA (chọn theo ABI thật)
import os, subprocess, sys
from google.colab import drive

print("🔗 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Dò tìm thư mục wheels trên Google Drive (hỗ trợ cả cấu trúc mirror GG_1):
WHEEL_CANDIDATES = [
    "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels",
    "/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels",
    "/content/drive/MyDrive/wheels",
]
WHEEL_DIR = None
for d in WHEEL_CANDIDATES:
    if os.path.isdir(d):
        WHEEL_DIR = d
        print(f"🎯 Đã tìm thấy thư mục wheels tại: {WHEEL_DIR}")
        break

assert WHEEL_DIR is not None, "❌ Không tìm thấy thư mục wheels trên Google Drive! Hãy đảm bảo thư mục wheels đã được tải lên Drive."

r = subprocess.run([sys.executable, "-c",
                    "import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')"],
                   capture_output=True, text=True)
abi = r.stdout.strip()
assert abi in ("TRUE", "FALSE"), f"❌ Không đọc được ABI: {r.stderr}"
print(f"🔍 ABI của torch đã cài: cxx11abi{abi}")

CAUSAL_WHL = f"{WHEEL_DIR}/causal_conv1d-1.7.0+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"
MAMBA_WHL  = f"{WHEEL_DIR}/mamba_ssm-2.3.2.post1+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"

for p in (CAUSAL_WHL, MAMBA_WHL):
    assert os.path.exists(p), f"❌ Không thấy wheel: {p}"

print("\n🚀 Đang cài 2 wheel từ Drive...")
!python -m pip install "{CAUSAL_WHL}" --force-reinstall --no-deps -q
!python -m pip install "{MAMBA_WHL}" --force-reinstall --no-deps -q

print("\n✅ CELL 3 XONG. Chạy tiếp cell 4 (khởi động lại runtime).")

In [ ]:
# 4. Khởi động lại Runtime để nạp thư viện C++/CUDA mới vào Python kernel
# ⚠️ LƯU Ý QUAN TRỌNG:
# Khi chạy ô này, Google Colab sẽ hiện thông báo đỏ: 'Your session crashed and has recovered'.
# Đây là điều HOÀN TOÀN BÌNH THƯỜNG và cần thiết để khởi động lại kernel nạp C++ CUDA extensions.
# Bạn chỉ cần ấn chạy tiếp Cell 5 ngay sau đó!
import os
os.kill(os.getpid(), 9)

In [ ]:
# 5. Kiểm tra causal_conv1d + mamba_ssm (xuôi + ngược, fp32 + fp16)
import sys, traceback, torch

print("python :", f"cp{sys.version_info.major}{sys.version_info.minor}")
print("torch  :", torch.__version__, "| ABI =", torch._C._GLIBCXX_USE_CXX11_ABI)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
import triton
print("triton :", triton.__version__, "| set_allocator:", hasattr(triton, "set_allocator"))

ok = {}

# --- Causal-Conv1d ---
try:
    import causal_conv1d, causal_conv1d_cuda
    from causal_conv1d import causal_conv1d_fn
    for dt in (torch.float32, torch.float16):
        x = torch.randn(2, 16, 64, device="cuda", dtype=dt, requires_grad=True)
        w = torch.randn(16, 4, device="cuda", dtype=dt, requires_grad=True)
        y = causal_conv1d_fn(x, w, activation="silu")
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(x.grad).all()
    ok["causal_conv1d"] = True
    print("causal_conv1d : ✅ xuôi + ngược, fp32 + fp16")
except Exception:
    ok["causal_conv1d"] = False
    print("causal_conv1d : ❌"); traceback.print_exc()

# --- Mamba-SSM ---
try:
    import mamba_ssm, selective_scan_cuda
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    B, D, L, N = 2, 16, 64, 8
    for dt in (torch.float32, torch.float16):
        u     = torch.randn(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        delta = torch.rand(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        A     = (-torch.rand(D, N, device="cuda")).requires_grad_()
        Bm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Cm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Dv    = torch.randn(D, device="cuda")
        y = selective_scan_fn(u, delta, A, Bm, Cm, D=Dv, delta_softplus=True)
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(u.grad).all()
    ok["mamba_ssm"] = True
    print("mamba_ssm     : ✅ selective_scan xuôi + ngược, fp32 + fp16")
except Exception:
    ok["mamba_ssm"] = False
    print("mamba_ssm     : ❌"); traceback.print_exc()

assert all(ok.values()), "❌ Kiểm tra mamba_ssm thất bại! Vui lòng kiểm tra lại log."
print("\n==> 🎉 MÔI TRƯỜNG CUDA MAMBA-SSM ĐÃ HOÀN TOÀN SẴN SÀNG!")

In [ ]:
# 6. Tải và đồng bộ mã nguồn từ GitHub
import os, sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('mamba_unet') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 7. Chuẩn bị dữ liệu từ Google Drive vào SSD cục bộ (/content/data/cache_1mm)
# Thư mục GG_1 trên máy tính được mirror trực tiếp vào Google Drive.
# Mã sẽ tự động tìm kiếm file zip theo đúng cấu trúc mirror của bạn:
import os
from common.data_pipeline.runtime_data import run_colab

DRIVE_ZIP_CANDIDATES = [
    # 1. Mirror trực tiếp thư mục GG_1 vào MyDrive:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 2. Mirror có thư mục cha GG_1 trong MyDrive:
    '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 3. File zip tại thư mục KAGGLE_UPLOAD:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/KAGGLE_UPLOAD/cache_1mm.zip',
    # 4. File zip đặt ngay tại gốc MyDrive:
    '/content/drive/MyDrive/cache_1mm.zip',
]

DRIVE_ZIP_PATH = None
for cand in DRIVE_ZIP_CANDIDATES:
    if os.path.exists(cand):
        DRIVE_ZIP_PATH = cand
        print(f'🎯 Đã tìm thấy file zip: {DRIVE_ZIP_PATH}')
        break

if DRIVE_ZIP_PATH is None:
    print('⚠️ Chưa thấy ở các đường dẫn cố định, sẽ tự động dò tìm sâu dưới /content/drive...')
    DRIVE_ZIP_PATH = 'auto'

# NUM_PATIENTS:
# - 0   : Giải nén toàn bộ 1010 bệnh nhân (cần ~40-60GB SSD Colab).
# - > 0 : Chọn N bệnh nhân đầu tiên (vd: 50, 100, 200) để huấn luyện nhanh và tiết kiệm dung lượng đĩa.
NUM_PATIENTS = 0

cache_dir = run_colab(
    zip_path=DRIVE_ZIP_PATH,
    dest_dir='/content/data/cache_1mm',
    n_patients=NUM_PATIENTS,
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

In [ ]:
# 8. Cấu hình chi tiết & Bắt đầu huấn luyện mô hình Mamba-UNet
import os

# =============================================================================
# BẢNG THAM SỐ TÙY CHỈNH (Tùy chỉnh trực tiếp các giá trị bên dưới)
# =============================================================================
# 1. Kiến trúc mô hình & Backbone MambaVision
BACKBONE = "mamba_vision_T"          # Lựa chọn: 'mamba_vision_T', 'mamba_vision_T2', 'mamba_vision_S', 'mamba_vision_B'
PRETRAINED = True                   # True: Dùng trọng số ImageNet-1K pretrained; False: Khởi tạo ngẫu nhiên
FREEZE_BACKBONE_EPOCHS = 5          # Số epoch đầu đóng băng backbone (-1: đóng băng suốt, 0: không đóng băng)
BACKBONE_LR_MULT = 0.1              # Hệ số nhân learning rate cho backbone so với decoder

# 2. Hàm kích hoạt Decoder (Activation)
# Lựa chọn: 'relu', 'gelu', 'silu', 'mish', 'leaky_relu', 'star_relu', 'soft_clamp_relu', 'soft_clamp_silu', 'identity'
ACTIVATION = "relu"

# 3. Hàm mất mát (Loss Function)
# Lựa chọn: 'dice_bce', 'dice', 'bce', 'tversky', 'focal_tversky', 'focal'
LOSS = "dice_bce"
LOSS_PARAMS = '{"bce_weight": 0.5, "dice_weight": 0.5}'  # Chuỗi JSON tham số loss, hoặc để trống ''

# 4. Siêu tham số huấn luyện (Training Hyperparameters)
EPOCHS = 50                         # Tổng số epoch huấn luyện
BATCH_SIZE = 8                      # Kích thước batch (khuyến nghị 8 trên Colab T4 16GB)
LR = 1e-4                           # Tốc độ học cơ sở (Learning rate)
WEIGHT_DECAY = 1e-4                 # Hệ số suy giảm trọng số AdamW
SCHEDULER = "cosine"                # Lựa chọn: 'cosine', 'plateau', 'step', 'none'
GRAD_CLIP = 1.0                     # Ngưỡng kẹp gradient tối đa (0 = tắt)
PATIENCE = 12                       # Số epoch chờ Early Stopping
WORKERS = 2                         # Số luồng nạp dữ liệu DataLoader (Colab khuyến nghị 2-4)

# 5. Quản lý phiên chạy & Khôi phục (Resume)
# RESUME: None (chạy mới từ đầu), 'auto' (tiếp tục phiên gần nhất bị ngắt), hoặc đường dẫn checkpoint/run cụ thể
RESUME = None                       # vd: 'auto' hoặc 'mamba_unet/runs/run_xxx/last_model.pth'
INIT_WEIGHTS = None                 # Nạp trọng số ban đầu nhưng tạo run mới (None nếu không dùng)
RUN_NAME = "mamba_colab_run"         # Tên phiên chạy (để trống '' để tự sinh theo timestamp)
TAG = "colab_mamba"                 # Thẻ phân loại thí nghiệm
NOTES = "Huấn luyện Mamba-UNet với kernel CUDA native selective_scan trên Colab"
SEED = 42

# 6. Tối ưu phần cứng & Chuẩn bị dữ liệu
FORCE_CUDA_BACKEND = True           # Ép dùng kernel CUDA native (DUNEAI_MAMBA_BACKEND=cuda)
USE_AMP = True                      # Bật Automatic Mixed Precision fp16 tăng tốc
GPU_WINDOW = False                  # Chuyển HU int16 lên GPU chuẩn hóa cửa sổ
NO_LUNG_MASK = False                # True: Không áp mặt nạ phổi ngoài lát cắt
NO_AUGMENT = False                  # True: Tắt lật/xoay ngẫu nhiên
LIMIT_TRAIN = 0                     # Giới hạn số lát cắt train (0 = toàn bộ)
LIMIT_VAL = 0                       # Giới hạn số lát cắt val (0 = toàn bộ)

# =============================================================================
# TỔ HỢP LỆNH HUẤN LUYỆN
# =============================================================================
env_prefix = "DUNEAI_MAMBA_BACKEND=cuda " if FORCE_CUDA_BACKEND else ""
cmd_parts = [
    f"{env_prefix}python -m mamba_unet.train_main train",
    f"--cache-dir /content/data/cache_1mm",
    f"--backbone {BACKBONE}",
    f"{'--pretrained' if PRETRAINED else '--no-pretrained'}",
    f"--freeze-backbone-epochs {FREEZE_BACKBONE_EPOCHS}",
    f"--backbone-lr-mult {BACKBONE_LR_MULT}",
    f"--activation {ACTIVATION}",
    f"--loss {LOSS}",
    f"--epochs {EPOCHS}",
    f"--batch-size {BATCH_SIZE}",
    f"--lr {LR}",
    f"--weight-decay {WEIGHT_DECAY}",
    f"--scheduler {SCHEDULER}",
    f"--grad-clip {GRAD_CLIP}",
    f"--patience {PATIENCE}",
    f"--workers {WORKERS}",
    f"--seed {SEED}",
]
if LOSS_PARAMS and LOSS_PARAMS.strip():
    cmd_parts.append(f"--loss-params '{LOSS_PARAMS.strip()}'")
if RESUME:
    cmd_parts.append(f"--resume {RESUME}")
if INIT_WEIGHTS:
    cmd_parts.append(f"--init-weights '{INIT_WEIGHTS}'")
if RUN_NAME:
    cmd_parts.append(f"--run-name '{RUN_NAME}'")
if TAG:
    cmd_parts.append(f"--tag '{TAG}'")
if NOTES:
    cmd_parts.append(f"--notes '{NOTES}'")
if not USE_AMP:
    cmd_parts.append("--no-amp")
if GPU_WINDOW:
    cmd_parts.append("--gpu-window")
if NO_LUNG_MASK:
    cmd_parts.append("--no-lung-mask")
if NO_AUGMENT:
    cmd_parts.append("--no-augment")
if LIMIT_TRAIN > 0:
    cmd_parts.append(f"--limit-train {LIMIT_TRAIN}")
if LIMIT_VAL > 0:
    cmd_parts.append(f"--limit-val {LIMIT_VAL}")

full_train_cmd = " \
    ".join(cmd_parts)
print("🚀 Đang khởi động tiến trình huấn luyện...")
print(f"Lệnh thực thi:\n{full_train_cmd}\n" + "=" * 80)
!{full_train_cmd}

In [ ]:
# 9. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
!python -m mamba_unet.evaluate eval \
    --cache-dir /content/data/cache_1mm \
    --checkpoint checkpoints/best_model.pth \
    --split val

In [ ]:
# 10. Lưu kết quả và Checkpoint vào Google Drive theo từng thư mục phiên chạy
# Tự động tạo mới hoặc cập nhật nối thêm (append) file REPORT.md và JSON tổng kết
import datetime, glob, json, os, shutil
from pathlib import Path

# 1. Xác định thư mục mirror Google Drive
MIRROR_SOURCES_DIR = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/DuneAI_Backups'

# 2. Tìm thư mục phiên chạy gần nhất trong mamba_unet/runs
local_runs = sorted(glob.glob("mamba_unet/runs/*"), key=os.path.getmtime)
if local_runs:
    active_local_run = Path(local_runs[-1])
    run_folder_name = active_local_run.name
else:
    run_folder_name = f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    active_local_run = Path(f"mamba_unet/runs/{run_folder_name}")
    active_local_run.mkdir(parents=True, exist_ok=True)

# 3. Thư mục đích trên Google Drive
drive_run_dir = Path(MIRROR_SOURCES_DIR) / "mamba_unet" / "runs" / run_folder_name
drive_ckpt_dir = Path(MIRROR_SOURCES_DIR) / "checkpoints"
drive_run_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

# 4. Trích xuất thông tin kết quả huấn luyện
now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
best_val_dice = "N/A"
best_epoch = "N/A"
if (active_local_run / "train_log.csv").exists():
    try:
        import pandas as pd
        df_log = pd.read_csv(active_local_run / "train_log.csv")
        if "val_dice" in df_log.columns and len(df_log) > 0:
            best_idx = df_log["val_dice"].idxmax()
            best_val_dice = f"{df_log.loc[best_idx, 'val_dice']:.4f}"
            best_epoch = f"{int(df_log.loc[best_idx, 'epoch'])}"
    except Exception:
        pass

# 5. Soạn thảo / Cập nhật file báo cáo REPORT.md
report_md_path = drive_run_dir / "REPORT.md"
is_resumed_session = bool(RESUME) and report_md_path.exists()

if is_resumed_session:
    print(f"🔄 Đang ghi đè/nối thêm (append) thông tin phiên Resume vào: {report_md_path}")
    resume_block = f"""
---
### 🔄 Phiên Tiếp Tục Huấn Luyện (Resume Session) - {now_str}
- **Khôi phục từ**: `{RESUME}`
- **Số Epochs đích**: {EPOCHS} | **Batch Size**: {BATCH_SIZE} | **Learning Rate**: {LR}
- **Scheduler**: {SCHEDULER} | **Workers**: {WORKERS}
- **Best Val Dice hiện tại**: **{best_val_dice}** (tại Epoch {best_epoch})
- **Ghi chú**: {NOTES}
"""
    with open(report_md_path, "a", encoding="utf-8") as f:
        f.write(resume_block)
else:
    print(f"📝 Đang tạo mới file báo cáo tổng hợp: {report_md_path}")
    new_report = f"""# 📋 Báo Cáo Huấn Luyện Mamba-UNet: {run_folder_name}

- **Thời gian khởi tạo**: {now_str}
- **Kiến trúc mô hình**: Mamba-UNet
  - **Backbone**: `{BACKBONE}` (Pretrained: `{PRETRAINED}`)
  - **Đóng băng Backbone**: {FREEZE_BACKBONE_EPOCHS} epochs đầu (LR mult: {BACKBONE_LR_MULT})
  - **Decoder Activation**: `{ACTIVATION}`
- **Hàm mất mát (Loss)**: `{LOSS}` (Tham số: `{LOSS_PARAMS}`)
- **Siêu tham số tối ưu**:
  - **Optimizer**: AdamW (LR: `{LR}`, Weight Decay: `{WEIGHT_DECAY}`)
  - **LR Scheduler**: `{SCHEDULER}`
  - **Gradient Clip**: `{GRAD_CLIP}` | **Patience (Early Stopping)**: `{PATIENCE}`
- **Cấu hình DataLoader**:
  - **Batch Size**: {BATCH_SIZE} | **Workers**: {WORKERS} | **Seed**: {SEED}
- **Backend Mamba**: CUDA Native Selective Scan (`mamba_ssm` + `causal_conv1d`)
- **Kết quả huấn luyện**:
  - **Best Val Dice**: **{best_val_dice}** (tại Epoch {best_epoch})
- **Ghi chú**: {NOTES}
"""
    with open(report_md_path, "w", encoding="utf-8") as f:
        f.write(new_report)

# Sao lưu báo cáo vào thư mục cục bộ và lưu thêm bản JSON
shutil.copy2(report_md_path, active_local_run / "REPORT.md")
report_json_data = {
    "run_folder": run_folder_name,
    "timestamp": now_str,
    "arch": "mamba_unet",
    "backbone": BACKBONE,
    "pretrained": PRETRAINED,
    "activation": ACTIVATION,
    "loss": LOSS,
    "loss_params": LOSS_PARAMS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "scheduler": SCHEDULER,
    "workers": WORKERS,
    "best_val_dice": best_val_dice,
    "best_epoch": best_epoch,
    "resume": RESUME,
    "notes": NOTES,
}
with open(drive_run_dir / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(report_json_data, f, indent=2, ensure_ascii=False)
shutil.copy2(drive_run_dir / "training_report.json", active_local_run / "training_report.json")

# 6. Sao chép toàn bộ checkpoints và artifacts của phiên chạy vào Drive
print(f"🚀 Đang sao lưu toàn bộ files từ {active_local_run} sang {drive_run_dir}...")
!cp -ru {active_local_run}/* '{drive_run_dir}/' 2>/dev/null || true
!cp -ru checkpoints/* '{drive_ckpt_dir}/' 2>/dev/null || true

print("=" * 80)
print(f"✅ HOÀN TẤT SAO LƯU!")
print(f"📁 Thư mục phiên chạy tại Google Drive: {drive_run_dir}")
print(f"📄 Báo cáo chi tiết: {report_md_path}")
print(f"🎯 Google Drive Desktop sẽ tự động đồng bộ thư mục này về máy tính của bạn!")
print("=" * 80)